Qwen3.5 From Scratch

In [ ]:
# 中文注释:检查本笔记本依赖的第三方库版本(huggingface_hub 用于下载预训练权重、
# tokenizers 用于加载/运行分词器、torch 用于搭建和运行模型本身)。
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 中文注释:选择要加载的 Qwen3.5 预训练权重版本,后面下载权重、配置模型维度
# 时都会引用这个字符串(本笔记本实际固定使用 0.8B 版本的配置)。
USE_MODEL = "Qwen3.5-0.8B"

1. Architecture code

In [ ]:
# ===========================================================================
# 中文注释(本单元总览):本单元定义 Qwen3.5 的完整模型结构。Qwen3.5(即
# Qwen3-Next 系列架构)是一个"混合(hybrid)"架构:大部分层是计算/显存随
# 序列长度线性增长的"线性注意力"层(Gated DeltaNet,门控增量法则),少数层
# (每 4 层里有 1 层,见后面 QWEN3_5_CONFIG["layer_types"])仍是标准的因果
# 自注意力(下面的 GroupedQueryAttention,即 full_attention)。
# 结构速览:
#   FeedForward      —— SwiGLU 前馈网络(所有层通用)
#   RMSNorm          —— 均方根归一化(所有层通用,替代 LayerNorm)
#   compute_rope_params / apply_rope —— 旋转位置编码(仅 full_attention 层使用)
#   GroupedQueryAttention —— 分组查询注意力(full_attention 层的 token mixer)
#   Qwen3_5GatedDeltaNet  —— 从 qwen3_5_transformers.py 导入,门控 DeltaNet
#                            线性注意力(linear_attention 层的 token mixer)
#   TransformerBlock —— 按 layer_type 在上面两种 token mixer 之间切换的
#                       预归一化(pre-norm)残差块
#   Qwen3_5Model     —— 堆叠 TransformerBlock、维护 KV 缓存的顶层模型
#   KVCache / Qwen3_5LinearAttentionCache —— 两种性质不同的缓存容器
#     (full_attention 层用随序列增长的 (k, v) 元组缓存;
#      linear_attention 层用大小恒定、不随序列增长的循环状态缓存)
# ===========================================================================
import torch
import torch.nn as nn


# 中文注释:SwiGLU 前馈网络(Qwen/Llama 系列模型的标准 FFN 结构)。
# fc1(gate_proj)和 fc2(up_proj)都把 emb_dim 投影到 hidden_dim,
# 用 SiLU(fc1(x)) * fc2(x) 做门控后,再由 fc3(down_proj)投影回 emb_dim。
# 注意:这里 fc1/fc2 的 in/out 维度写成 (emb_dim, hidden_dim),
# 与常见实现按 nn.Linear(in_features, out_features) 的含义一致。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # 前向传播形状:x: [batch, seq_len, emb_dim]
        #   x_fc1, x_fc2: [batch, seq_len, hidden_dim]
        #   silu(x_fc1) * x_fc2: [batch, seq_len, hidden_dim](逐元素门控)
        #   最终输出: [batch, seq_len, emb_dim]
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)
# 中文注释:RMSNorm(均方根归一化)。与 LayerNorm 不同,RMSNorm 不减均值,
# 只用"均方根"(root mean square)做缩放,计算更简单、经验上效果相近。
# Qwen3.5 这里用的是 "weight 初始化为全零 + (1 + weight) 缩放" 的写法,
# 好处是模型刚初始化时 (1 + 0) = 1,即 RMSNorm 退化成"纯归一化、不做
# 额外缩放",让残差分支在训练初期更接近恒等映射,有利于深层网络的训练稳定性。
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Qwen3.5 uses (1 + weight) scaling with zero init
        self.weight = nn.Parameter(torch.zeros(emb_dim))

    # x 形状: [..., emb_dim];在最后一维(emb_dim)上计算均方根并归一化,
    # 归一化后形状不变。
    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

    # 前向传播先转 float32 计算(_norm 内部用到的 rsqrt 对精度敏感,
    # bf16/fp16 下直接算容易损失精度甚至数值不稳定),
    # 乘以 (1 + weight) 做缩放后,再转换回输入原本的 dtype(如 bfloat16)。
    def forward(self, x):
        x_norm = self._norm(x.float())
        x_norm = x_norm * (1.0 + self.weight.float())
        return x_norm.to(dtype=x.dtype)
# 中文注释:构造 RoPE(旋转位置编码,Rotary Position Embedding)用到的
# cos/sin 查找表。
# - theta_base 控制旋转频率的"基数"(Qwen3.5 用 1000万这个很大的值,配合
#   下面的 partial_rotary_factor,是为了更好地支持超长上下文,本模型
#   context_length 高达 262144)。
# - partial_rotary_factor(Qwen3.5 取 0.25):并不是对整个 head_dim 都施加
#   旋转,而只对 head_dim 的前 rotary_dim = head_dim * partial_rotary_factor
#   个维度施加 RoPE,其余维度(pass-through)不参与旋转、原样保留
#   (具体见下面 apply_rope 中的 x_pass)。这是 Qwen3.5/Qwen3-Next 的
#   "部分旋转位置编码"设计。
def compute_rope_params(
    head_dim,
    theta_base=10_000,
    context_length=4096,
    partial_rotary_factor=1.0,
    dtype=torch.float32,
):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # rotary_dim 需要是偶数(RoPE 是把维度两两一组做旋转),
    # 这里向下取整到最近的偶数。
    rotary_dim = int(head_dim * partial_rotary_factor)
    rotary_dim = max(2, rotary_dim - (rotary_dim % 2))

    # inv_freq 形状: [rotary_dim // 2],每一维对应一个不同的旋转频率,
    # 频率随维度指数衰减(标准 Transformer 正弦位置编码的频率构造方式)。
    inv_freq = 1.0 / (
        theta_base ** (
            torch.arange(0, rotary_dim, 2, dtype=dtype)[: (rotary_dim // 2)].float() / rotary_dim
        )
    )

    # positions: [context_length],角度 = 位置 × 频率。
    # angles 形状: [context_length, rotary_dim // 2],
    # 拼接自身后变成 [context_length, rotary_dim](把同一组频率复制一份,
    # 分别用于"旋转半径"公式里的前半段和后半段,见 apply_rope)。
    positions = torch.arange(context_length, dtype=dtype)
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    angles = torch.cat([angles, angles], dim=1)

    # cos, sin 最终形状均为: [context_length, rotary_dim],
    # 作为查找表在 apply_rope 中按位置切片使用。
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 中文注释:把 RoPE 旋转应用到 query/key 张量上。
# x 形状: [batch, num_heads, seq_len, head_dim]。
# 只旋转前 rot_dim(= compute_rope_params 中的 rotary_dim)个维度
# (x_rot),剩余的 head_dim - rot_dim 维(x_pass)保持不变、直接拼回去。
# offset 参数用于支持增量解码(KV 缓存):当输入不是从序列开头算起时
# (比如只喂入新 token,或者要重新计算历史 key 的位置编码),需要知道这段
# 输入在完整序列中的绝对起始位置,才能从 cos/sin 表中取到正确的行。
def apply_rope(x, cos, sin, offset=0):
    _, _, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    rot_dim = cos.shape[-1]
    # rot_dim 由传入的 cos 表的最后一维决定(即 compute_rope_params 里的
    # rotary_dim);必须不超过 head_dim,否则说明配置有误。
    if rot_dim > head_dim:
        raise ValueError(f"RoPE dim {rot_dim} cannot exceed head_dim {head_dim}.")

    # 按最后一维(head_dim)切开:x_rot 是要施加旋转的部分,
    # x_pass 是"部分旋转"设计里不参与旋转、原样透传的部分。
    x_rot = x[..., :rot_dim]
    x_pass = x[..., rot_dim:]

    # 标准 RoPE 的"旋转一半"(rotate half)技巧:把 rot_dim 切成前后两半
    # (x1, x2),后面用 (-x2, x1) 拼接构造出与 [x1, x2] 正交的向量,
    # 从而实现二维平面上的旋转变换: [x1,x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]。
    x1 = x_rot[..., : rot_dim // 2]
    x2 = x_rot[..., rot_dim // 2 :]

    # 从预先算好的 cos/sin 表中,取出 [offset, offset+seq_len) 这一段位置
    # 对应的行(形状 [seq_len, rot_dim]),再 unsqueeze 出 batch 和 head 维,
    # 方便与 x_rot([batch, num_heads, seq_len, rot_dim])做广播乘法。
    cos = cos[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[offset:offset + seq_len, :].unsqueeze(0).unsqueeze(0)

    # rotated = (-x2, x1) 拼接,即上面提到的"旋转一半"技巧;
    # x_rotated = x_rot * cos + rotated * sin,就是完整的 RoPE 旋转公式。
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x_rot * cos) + (rotated * sin)

    x_out = torch.cat([x_rotated, x_pass], dim=-1)
    return x_out.to(dtype=x.dtype)
# 中文注释:GroupedQueryAttention —— 本模型里 layer_type == "full_attention"
# 的层所使用的 token mixer(标准因果自注意力 + 分组查询注意力 GQA)。
# 相对基础版 GQA,这里有两个 Qwen3 系列特有的改动:
#   1) qk_norm:对 query/key 在"拆分出每个 head"之后、"施加 RoPE"之前,
#      按 head_dim 做一次 RMSNorm,用来稳定注意力打分的数值范围;
#   2) 门控 Q 投影:W_query 的输出维度是常规 d_out 的 2 倍,一半当 query,
#      另一半当 sigmoid 门控,在注意力输出(context)上做逐元素缩放
#      (而不是像常规注意力那样直接输出 context)。
# 另外,这里 head_dim 由配置显式指定(256),与 d_in // num_heads
# (1024 // 8 = 128)并不相等,因此 d_out = num_heads * head_dim(2048)
# 也不等于 d_in(1024),需要靠 out_proj 把维度投影回 d_in。
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # Qwen3.5 full-attention uses a gated Q projection (2x output dim)
        self.W_query = nn.Linear(d_in, self.d_out * 2, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    # 形状说明:
    #   x: [batch, num_tokens, d_in]
    #   mask: 因果掩码,形状 [1, 1, num_tokens(或增量步数), 总 kv 长度]
    #   cos, sin: 来自 Qwen3_5Model 预先算好的 RoPE 查找表
    #   start_pos: 本次前向的这批 token,在整段序列中的绝对起始位置
    #     (prefill 时为 0;增量解码时为 KV 缓存里已有的 token 数)
    #   cache: None,或本层上一次前向返回的 (prev_k, prev_v) 元组,
    #     形状均为 [batch, n_kv_groups, prev_len, head_dim](未施加 RoPE 的
    #     "原始" key/value,见下方注释)
    def forward(self, x, mask, cos, sin, start_pos=0, cache=None):
        b, num_tokens, _ = x.shape

        # W_query 输出维度是 d_out 的 2 倍(见类注释里的"门控 Q 投影"),
        # view 成 [b, num_tokens, num_heads, head_dim * 2] 后,
        # 用 chunk 沿最后一维一分为二:queries 是真正参与注意力计算的部分,
        # gate 是稍后要施加 sigmoid 的门控信号,形状都还原成 [.., head_dim]/
        # 展平回 [b, num_tokens, d_out]。
        q_and_gate = self.W_query(x)
        q_and_gate = q_and_gate.view(b, num_tokens, self.num_heads, self.head_dim * 2)
        queries, gate = torch.chunk(q_and_gate, 2, dim=-1)
        gate = gate.reshape(b, num_tokens, self.d_out)

        # keys/values 走 num_kv_groups(比 num_heads 少)份独立投影,
        # 形状均为 [b, num_tokens, num_kv_groups * head_dim]
        # ——这就是分组查询注意力(GQA)节省 KV 缓存显存的关键:
        # 缓存的是 num_kv_groups 份而不是 num_heads 份 key/value。
        keys = self.W_key(x)
        values = self.W_value(x)

        # 转置成 [batch, heads, seq, head_dim] 布局:
        # queries -> [b, num_heads, num_tokens, head_dim]
        # keys_new/values_new -> [b, num_kv_groups, num_tokens, head_dim]
        queries = queries.transpose(1, 2)
        keys_new = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values_new = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # qk_norm(Qwen3 系列特有):在"拆好 head、还未施加 RoPE"的阶段,
        # 对每个 head 的最后一维(head_dim)做 RMSNorm,注意顺序——
        # 先归一化,再在下面对归一化后的结果施加旋转位置编码。
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys_new = self.k_norm(keys_new)

        # 中文注释:把这一步新算出的 keys_new/values_new,与 KV 缓存中
        # 保存的历史 key/value(prev_k, prev_v,均为"尚未施加 RoPE 的原始值")
        # 拼接成完整的 keys_cat_raw/values_cat_raw,用于下面计算注意力。
        # 注意:下面会对拼接后的"整段历史" keys 重新施加一次 RoPE
        # (而不是缓存"已加过 RoPE 的 key"),所以缓存里存的必须是
        # 旋转之前的原始 key/value——这也是为什么下面 next_cache 用的是
        # keys_new/values_new 而不是加过 RoPE 之后的 keys。
        prev_len = 0
        if cache is not None:
            prev_k, prev_v = cache
            if prev_k is not None:
                prev_len = prev_k.size(2)
                keys_cat_raw = torch.cat([prev_k, keys_new], dim=2)
                values_cat_raw = torch.cat([prev_v, values_new], dim=2)
            else:
                keys_cat_raw = keys_new
                values_cat_raw = values_new
        else:
            keys_cat_raw = keys_new
            values_cat_raw = values_new

        # 对 query 施加 RoPE:offset=start_pos,即 query 自己在整段序列中的
        # 绝对位置。
        # 对(拼接后的全部历史)key 施加 RoPE:offset=start_pos - prev_len,
        # 即这段拼接后的 key 序列中,第 0 个位置对应的绝对位置
        # (prev_len 是历史 key 的长度,start_pos - prev_len 正好是历史
        # 部分第一个 token 的绝对位置)。这里的代价是:每次增量解码都要
        # 对全部历史 key 重新计算一遍 RoPE(而不是只缓存旋转后的 key),
        # 计算量随缓存长度线性增长,属于"用少量重复计算换实现简洁"的
        # 工程取舍,不是数值上的错误。
        queries = apply_rope(queries, cos, sin, offset=start_pos)
        keys = apply_rope(keys_cat_raw, cos, sin, offset=start_pos - prev_len)

        # GQA 的核心操作:把 num_kv_groups 份 key/value,沿 head 维度
        # 各重复 group_size(= num_heads // num_kv_groups)次,
        # 从而把 head 数扩充到与 query 的 num_heads 对齐,
        # 这样才能做逐 head 的批量矩阵乘法。
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values_cat_raw.repeat_interleave(self.group_size, dim=1)

        # 构造要返回给调用方、写入 KV 缓存的 next_cache:同样是把
        # "未加 RoPE 的原始" keys_new/values_new 拼接到旧缓存后面
        # (与上面 keys_cat_raw/values_cat_raw 的拼接结果等价,只是
        # 这里是单独重新计算的一份,用于返回)。
        if cache is not None and cache[0] is not None:
            next_cache = (
                torch.cat([cache[0], keys_new], dim=2),
                torch.cat([cache[1], values_new], dim=2),
            )
        else:
            next_cache = (keys_new, values_new)

        # 标准缩放点积注意力:
        # attn_scores: [b, num_heads, num_tokens, 总kv长度]
        # 用 -inf 填充 mask 标记的位置(因果掩码,禁止看到未来 token),
        # softmax 显式用 float32 计算以保证数值稳定,算完再转回原 dtype。
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(
            attn_scores * (self.head_dim ** -0.5),
            dim=-1,
            dtype=torch.float32,
        ).to(queries.dtype)

        # context: [b, num_tokens, d_out],把各 head 的注意力输出拼回一起。
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)

        # 中文:门控 Q 投影的"落地"位置——用 sigmoid(gate) 对注意力输出
        # context 做逐元素缩放(而不是缩放 attention weights 或 query 本身),
        # 这是 Qwen3.5 full_attention 层相对基础 GQA 的改动点。
        # Qwen3.5 full-attention uses a gated Q projection
        context = context * torch.sigmoid(gate)
        out = self.out_proj(context)
        return out, next_cache
# 中文注释:从配套文件 qwen3_5_transformers.py(移植自 HuggingFace
# transformers 的 Qwen3.5/Qwen3-Next 建模代码)导入 Qwen3_5GatedDeltaNet,
# 即本模型里 layer_type == "linear_attention" 的层所使用的 token mixer。
# 该类内部实现了"深度可分离短因果卷积 + 门控增量法则(gated delta rule)"
# 的线性注意力机制,具体数学细节见 qwen3_5_transformers.py 中的详细注释,
# 这里只关注它在本模型中如何被接入、如何与 KV 缓存交互。
from qwen3_5_transformers import (
    Qwen3_5GatedDeltaNet,
)

# Just a mapping for the different naming convention in Hugging Face transformers
# 中文注释:_Qwen3_5ConfigAdapter 只是一个"字段改名"适配器——本笔记本的
# QWEN3_5_CONFIG 用的是扁平字典 + 简短 key(如 "emb_dim"),而
# Qwen3_5GatedDeltaNet(照搬自 HuggingFace transformers)期望的是一个
# 有 .hidden_size / .linear_num_value_heads 等属性的配置对象,
# 这个类就是把字典 key 映射成 HuggingFace 风格的属性名,不涉及任何
# 数值计算逻辑。
class _Qwen3_5ConfigAdapter:
    def __init__(self, cfg):
        self.hidden_size = cfg["emb_dim"]
        self.linear_num_value_heads = cfg["linear_num_value_heads"]
        self.linear_num_key_heads = cfg["linear_num_key_heads"]
        self.linear_key_head_dim = cfg["linear_key_head_dim"]
        self.linear_value_head_dim = cfg["linear_value_head_dim"]
        self.linear_conv_kernel_dim = cfg["linear_conv_kernel_dim"]
        self.hidden_act = "silu"
        self.rms_norm_eps = cfg.get("rms_norm_eps", 1e-6)
        self.dtype = cfg.get("dtype", None)


# 中文注释:TransformerBlock —— 预归一化(pre-norm)残差 Transformer 块,
# 根据 layer_type 在两种完全不同的 token mixer 之间切换:
#   full_attention   -> GroupedQueryAttention(标准注意力 + KV 缓存)
#   linear_attention -> Qwen3_5GatedDeltaNet(线性注意力 + 循环状态缓存)
# 两种 token mixer 的缓存机制完全不同:full_attention 通过函数参数
# cache 传入/返回 (k, v) 元组(next_cache 需要显式往上层传递并写回
# KVCache);linear_attention 则是直接持有一个共享的 linear_cache 对象
# (Qwen3_5LinearAttentionCache),在 forward 内部按 layer_idx 原地读写
# 其中的卷积状态/循环状态,不需要通过返回值传递,所以下面 next_cache
# 恒为 None。
class TransformerBlock(nn.Module):
    def __init__(self, cfg, layer_type, layer_idx):
        super().__init__()
        self.layer_type = layer_type

        if layer_type == "full_attention":
            self.token_mixer = GroupedQueryAttention(
                d_in=cfg["emb_dim"],
                num_heads=cfg["n_heads"],
                head_dim=cfg["head_dim"],
                num_kv_groups=cfg["n_kv_groups"],
                qk_norm=cfg["qk_norm"],
                dtype=cfg["dtype"],
            )
        elif layer_type == "linear_attention":
            self.token_mixer = Qwen3_5GatedDeltaNet(_Qwen3_5ConfigAdapter(cfg), layer_idx)
        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))

    # 形状说明:x: [batch, num_tokens, emb_dim];mask/cos/sin/start_pos/
    # cache 均直接透传给 full_attention 分支的 GroupedQueryAttention;
    # linear_cache 是整个模型共享的 Qwen3_5LinearAttentionCache 实例
    # (只有 linear_attention 分支会用到);cache_position 是本次前向
    # 这批 token 的绝对位置索引,用于 Qwen3_5GatedDeltaNet 内部判断
    # 是否处于"已有历史缓存 + 单 token 增量解码"模式。
    def forward(self, x, mask, cos, sin, start_pos=0, cache=None, linear_cache=None, cache_position=None):
        shortcut = x
        x = self.norm1(x)

        # full_attention 分支:标准预归一化自注意力,返回更新后的
        # (k, v) 缓存元组 next_cache,由上层 Qwen3_5Model 写回 KVCache。
        if self.layer_type == "full_attention":
            x, next_cache = self.token_mixer(
                x,
                mask,
                cos,
                sin,
                start_pos=start_pos,
                cache=cache,
            )
        else:
            # linear_attention 分支:调用 Qwen3_5GatedDeltaNet,
            # 状态(卷积窗口 + 循环状态)通过 linear_cache 原地读写,
            # 不需要函数返回值携带缓存,故 next_cache 设为 None。
            x = self.token_mixer(
                x,
                cache_params=linear_cache,
                cache_position=cache_position,
            )
            next_cache = None

        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut

        return x, next_cache
# 中文注释:Qwen3_5Model —— 顶层模型。
# 主要职责:
#   1) token embedding;
#   2) 按 layer_types(见 QWEN3_5_CONFIG,典型模式是"3 个 linear_attention
#      + 1 个 full_attention"循环 6 次,共 24 层)堆叠 TransformerBlock;
#   3) 预先计算好 RoPE 的 cos/sin 查找表并注册为 buffer
#      (注意:RoPE 只给 full_attention 层用,linear_attention 层
#      不使用位置编码,而是靠门控增量法则自身的衰减机制隐式建模位置信息);
#   4) 维护/驱动 KV 缓存(KVCache),支持逐 token 增量解码。
class Qwen3_5Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        # layer_types 未显式提供时,默认所有层都是 full_attention
        # (纯注意力模型,退化成普通 Transformer);Qwen3.5 实际配置里
        # 会显式提供混合模式列表(见后面 QWEN3_5_CONFIG)。
        layer_types = cfg.get("layer_types", ["full_attention"] * cfg["n_layers"])
        if len(layer_types) != cfg["n_layers"]:
            raise ValueError("len(layer_types) must equal n_layers")

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg, layer_type, idx) for idx, layer_type in enumerate(layer_types)]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        head_dim = cfg["emb_dim"] // cfg["n_heads"] if cfg["head_dim"] is None else cfg["head_dim"]
        # 注意:这里的 head_dim 是 full_attention 层的 head_dim,
        # RoPE 查找表也只按这个 head_dim(及 partial_rotary_factor 截出的
        # rotary_dim)构造;linear_attention 层有自己独立的
        # linear_key_head_dim/linear_value_head_dim,并不使用这张表。
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            partial_rotary_factor=cfg.get("partial_rotary_factor", 1.0),
            dtype=torch.float32,
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg
        self.current_pos = 0

    # 中文注释:构造标准的因果(causal)掩码——上三角(不含对角线)为 True
    # 表示"禁止看到未来",下三角(含对角线)为 False 表示"允许注意力"。
    # pos_start/pos_end 允许只截取某几行(对应本次新增的 token 在 query
    # 维度上的位置),但列维度始终是 [0, pos_end),即可以看到从序列开头到
    # 当前为止的所有 key——这正是支持"只喂入新 token、复用 KV 缓存"的
    # 增量解码所需要的矩形掩码。
    def create_mask(self, cur_len, device, pos_start=0, pos_end=None):
        if pos_end is None:
            pos_end = cur_len

        ones = torch.ones((pos_end, pos_end), device=device, dtype=torch.bool)
        mask_full = torch.triu(ones, diagonal=1)
        row_slice = slice(pos_start, pos_end)
        mask = mask_full[row_slice, :pos_end][None, None, :, :]
        return mask

    # 形状说明:in_idx: [batch, num_tokens](token id 序列);
    # x = self.tok_emb(in_idx): [batch, num_tokens, emb_dim]。
    def forward(self, in_idx, cache=None):
        x = self.tok_emb(in_idx)

        num_tokens = x.shape[1]
        # 使用 KV 缓存(增量解码/续写场景):self.current_pos 记录"到目前
        # 为止已经处理过多少个 token",每次前向按 [pos_start, pos_end)
        # 更新;cache_position 是这批新 token 的绝对位置索引,会一路传给
        # 每个 TransformerBlock 的 linear_attention 分支使用。
        # 注意:开始一次新的生成前必须先调用 reset_kv_cache() 把
        # current_pos 归零,否则位置会从上一次生成结束的地方继续累加。
        if cache is not None:
            pos_start = self.current_pos
            pos_end = pos_start + num_tokens
            self.current_pos = pos_end
            mask = self.create_mask(
                cur_len=num_tokens,
                device=x.device,
                pos_start=pos_start,
                pos_end=pos_end,
            )
            cache_position = torch.arange(pos_start, pos_end, device=x.device, dtype=torch.long)
        else:
            pos_start = 0
            mask = self.create_mask(
                cur_len=num_tokens,
                device=x.device,
                pos_start=0,
                pos_end=num_tokens,
            )
            cache_position = None

        # 依次跑每一层:blk_cache 是该层(仅对 full_attention 层有意义)
        # 自己的 (k, v) 缓存;cache.linear_cache 是整个模型共享的同一个
        # Qwen3_5LinearAttentionCache 实例,内部按 layer_idx 存取各
        # linear_attention 层各自的卷积/循环状态,因此无需每层单独取值,
        # 直接把整个对象传下去即可。
        for i, block in enumerate(self.trf_blocks):
            blk_cache = cache.get(i) if cache is not None else None
            x, new_blk_cache = block(
                x,
                mask=mask,
                cos=self.cos,
                sin=self.sin,
                start_pos=pos_start,
                cache=blk_cache,
                linear_cache=cache.linear_cache if cache is not None else None,
                cache_position=cache_position,
            )
            if cache is not None and new_blk_cache is not None:
                cache.update(i, new_blk_cache)

        # 本次前向(无论是首次 prefill 还是后续增量解码)结束后,
        # 标记 linear_cache 里"已经有历史状态"了——下一次调用时,
        # Qwen3_5GatedDeltaNet 就会依据 has_previous_state + seq_len==1
        # 判断进入"逐 token 递归"的增量解码路径,而不是"分块并行"路径。
        if cache is not None:
            cache.linear_cache.has_previous_state = True

        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

    # 开始新一轮生成前需要调用一次,把绝对位置计数器归零
    # (但注意:这个方法不会重置 KVCache/linear_cache 里保存的具体张量,
    # 真正的缓存内容清空要靠 KVCache.reset() / Qwen3_5LinearAttentionCache.reset())。
    def reset_kv_cache(self):
        self.current_pos = 0


# 中文注释:Qwen3_5LinearAttentionCache —— 专门给 linear_attention 层
# (Qwen3_5GatedDeltaNet)用的缓存,按层索引(layer_idx)存两类状态:
#   conv_states[i]:因果卷积的滑动窗口历史,形状约为
#     [batch, conv_dim, kernel_size - 1],与序列长度无关;
#   recurrent_states[i]:门控增量法则的循环状态矩阵,形状约为
#     [batch, num_v_heads, head_k_dim, head_v_dim],同样与序列长度无关。
# 这正是线性注意力相对标准注意力最大的优势:无论生成了多少个 token,
# 这里缓存的张量大小始终不变(O(1) 显存),而标准注意力的 KV 缓存
# 会随 token 数线性增长。
# has_previous_state 用来标记"是否已经跑过至少一次前向",决定
# Qwen3_5GatedDeltaNet 走分块并行还是逐 token 递归的计算路径。
class Qwen3_5LinearAttentionCache:
    def __init__(self, n_layers):
        self.conv_states = [None] * n_layers
        self.recurrent_states = [None] * n_layers
        self.has_previous_state = False

    def reset(self):
        for i in range(len(self.conv_states)):
            self.conv_states[i] = None
            self.recurrent_states[i] = None
        self.has_previous_state = False


# 中文注释:KVCache —— 顶层缓存容器,把两种性质不同的缓存组合在一起:
#   self.cache:按 layer_idx 存放 full_attention 层的 (k, v) 元组,
#     每个元素形状 [batch, n_kv_groups, 已生成的 token 数, head_dim],
#     会随生成的 token 数线性增长;
#   self.linear_cache:上面定义的 Qwen3_5LinearAttentionCache 实例,
#     大小恒定,供所有 linear_attention 层共享。
class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers
        self.linear_cache = Qwen3_5LinearAttentionCache(n_layers)

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None
        self.linear_cache.reset()

2. Initialize model

In [ ]:
# 中文注释:Qwen3.5-0.8B 的完整配置字典。几个关键点:
# - head_dim=256、n_heads=8、n_kv_groups=2:GQA 分组大小
#   group_size = n_heads // n_kv_groups = 4,即每 4 个 query head
#   共享同一组 key/value head;d_out = n_heads * head_dim = 2048,
#   与 emb_dim=1024 不同,靠 out_proj 投影回 emb_dim。
# - partial_rotary_factor=0.25:RoPE 只旋转 head_dim 的 25%
#   (256 * 0.25 = 64 维),其余 192 维原样透传,配合很大的
#   rope_base(1000万)支持 context_length 高达 262144。
# - linear_* 系列字段(linear_num_key_heads/linear_num_value_heads=16、
#   linear_key_head_dim/linear_value_head_dim=128、
#   linear_conv_kernel_dim=4)专门配置 linear_attention 层
#   (Qwen3_5GatedDeltaNet)的头数/头维度/短因果卷积核大小,
#   与上面 full_attention 层的头配置完全独立。
# - layer_types:24 层里每 4 层是"3 个 linear_attention + 1 个
#   full_attention"的重复模式,这就是 Qwen3.5/Qwen3-Next 的
#   混合架构设计——绝大多数层用开销随长度线性增长的线性注意力,
#   少数层保留标准注意力以保证建模能力。
# Qwen3.5-0.8B text configuration
QWEN3_5_CONFIG = {
    "vocab_size": 248_320,
    "context_length": 262_144,
    "emb_dim": 1_024,
    "n_heads": 8,
    "n_layers": 24,
    "hidden_dim": 3_584,
    "head_dim": 256,
    "qk_norm": True,
    "n_kv_groups": 2,
    "rope_base": 10_000_000.0,
    "partial_rotary_factor": 0.25,
    "rms_norm_eps": 1e-6,
    "linear_conv_kernel_dim": 4,
    "linear_key_head_dim": 128,
    "linear_value_head_dim": 128,
    "linear_num_key_heads": 16,
    "linear_num_value_heads": 16,
    "dtype": torch.bfloat16,
    "layer_types": [
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
    ],
}
# 中文注释:固定随机种子(此时模型参数仍是随机初始化的,真正的预训练
# 权重要等到后面 "3. Load pretrained weights" 一节才会被加载进来);
# 随后按上面的配置实例化 Qwen3_5Model。
torch.manual_seed(123)
model = Qwen3_5Model(QWEN3_5_CONFIG)

In [ ]:
model

In [ ]:
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 中文注释:统计模型总参数量。下面"减去 token embedding 参数量"是在
# 假设 tok_emb 与 out_head(LM head)权重绑定(weight tying)的前提下,
# 估算"去重后"的真实参数量——注意此时权重还未从 checkpoint 加载
# (真正是否绑定,要看后面 load_weights_into_qwen3_5 里检测到
# checkpoint 中是否存在独立的 lm_head.weight)。
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 中文注释:粗略估算模型占用的显存/内存大小(参数 + 假设同精度存储的
# 梯度 + buffer,例如 RoPE 的 cos/sin 表),按不同 dtype(float32 /
# bfloat16)分别换算成 GB,便于评估能否放进显卡显存。
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 中文注释:按优先级选择运行设备(CUDA GPU > Apple Silicon MPS > CPU),
# 并把模型搬到该设备上。
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

3. Load pretrained weights

In [ ]:
# 中文注释:把从 HuggingFace safetensors checkpoint 加载出来的原始权重
# (键名类似 "model.layers.{l}.self_attn.q_proj.weight"),逐个搬运/赋值
# 到我们自己搭建的 Qwen3_5Model 对应的子模块参数上。核心是按
# layer_types[l] 区分:full_attention 层搬运 self_attn.* 系列权重,
# linear_attention 层搬运 linear_attn.* 系列权重(对应 Qwen3_5GatedDeltaNet
# 里的 dt_bias/A_log/conv1d/norm/out_proj/in_proj_qkv/in_proj_z/
# in_proj_b/in_proj_a 等参数),两类层共享的 FFN 与两个 RMSNorm
# (input_layernorm / post_attention_layernorm)按相同方式搬运。
def load_weights_into_qwen3_5(model, param_config, params):
    # assign:一个带形状校验的赋值小工具——先检查形状是否一致
    # (避免因为写错权重 key 或 reshape 而悄悄错位),再用 copy_
    # 原地写入(不改变参数对象本身,只改变其数值),最后把该参数对象
    # 原样返回,方便写成 `model.xxx = assign(model.xxx, checkpoint_tensor)`
    # 这种链式写法。
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 不同来源的 checkpoint,embedding 权重可能在两种不同的键名前缀下
    # (纯语言模型 checkpoint 用 "model.",多模态模型的语言模型子模块
    # 用 "model.language_model."),这里做一次探测以兼容两种情况。
    if "model.embed_tokens.weight" in params:
        model_prefix = "model"
    elif "model.language_model.embed_tokens.weight" in params:
        model_prefix = "model.language_model"
    else:
        raise KeyError("Could not find embed token weights in checkpoint.")

    def pkey(suffix):
        return f"{model_prefix}.{suffix}"

    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params[pkey("embed_tokens.weight")],
        pkey("embed_tokens.weight"),
    )

    n_layers = param_config["n_layers"]
    layer_types = param_config.get("layer_types", ["full_attention"] * n_layers)

    for l in range(n_layers):
        block = model.trf_blocks[l]
        layer_type = layer_types[l]

        # full_attention 层:搬运标准自注意力的 q/k/v/o 投影权重,
        # 以及(若启用了 qk_norm)对应的 q_norm/k_norm RMSNorm 权重。
        if layer_type == "full_attention":
            att = block.token_mixer
            att.W_query.weight = assign(
                att.W_query.weight,
                params[pkey(f"layers.{l}.self_attn.q_proj.weight")],
                pkey(f"layers.{l}.self_attn.q_proj.weight"),
            )
            att.W_key.weight = assign(
                att.W_key.weight,
                params[pkey(f"layers.{l}.self_attn.k_proj.weight")],
                pkey(f"layers.{l}.self_attn.k_proj.weight"),
            )
            att.W_value.weight = assign(
                att.W_value.weight,
                params[pkey(f"layers.{l}.self_attn.v_proj.weight")],
                pkey(f"layers.{l}.self_attn.v_proj.weight"),
            )
            att.out_proj.weight = assign(
                att.out_proj.weight,
                params[pkey(f"layers.{l}.self_attn.o_proj.weight")],
                pkey(f"layers.{l}.self_attn.o_proj.weight"),
            )
            if hasattr(att, "q_norm") and att.q_norm is not None:
                att.q_norm.weight = assign(
                    att.q_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.q_norm.weight")],
                    pkey(f"layers.{l}.self_attn.q_norm.weight"),
                )
            if hasattr(att, "k_norm") and att.k_norm is not None:
                att.k_norm.weight = assign(
                    att.k_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.k_norm.weight")],
                    pkey(f"layers.{l}.self_attn.k_norm.weight"),
                )

        # linear_attention 层:搬运 Gated DeltaNet 的全部可训练参数——
        # dt_bias/A_log 控制门控衰减系数 g,conv1d 是短因果卷积的核,
        # norm 是门控 RMSNorm 的缩放权重,in_proj_qkv/in_proj_z/in_proj_b/
        # in_proj_a 是四路独立的输入投影,out_proj 是输出投影。
        elif layer_type == "linear_attention":
            lat = block.token_mixer
            lat.dt_bias = assign(
                lat.dt_bias,
                params[pkey(f"layers.{l}.linear_attn.dt_bias")],
                pkey(f"layers.{l}.linear_attn.dt_bias"),
            )
            lat.A_log = assign(
                lat.A_log,
                params[pkey(f"layers.{l}.linear_attn.A_log")],
                pkey(f"layers.{l}.linear_attn.A_log"),
            )
            lat.conv1d.weight = assign(
                lat.conv1d.weight,
                params[pkey(f"layers.{l}.linear_attn.conv1d.weight")],
                pkey(f"layers.{l}.linear_attn.conv1d.weight"),
            )
            lat.norm.weight = assign(
                lat.norm.weight,
                params[pkey(f"layers.{l}.linear_attn.norm.weight")],
                pkey(f"layers.{l}.linear_attn.norm.weight"),
            )
            lat.out_proj.weight = assign(
                lat.out_proj.weight,
                params[pkey(f"layers.{l}.linear_attn.out_proj.weight")],
                pkey(f"layers.{l}.linear_attn.out_proj.weight"),
            )
            lat.in_proj_qkv.weight = assign(
                lat.in_proj_qkv.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight"),
            )
            lat.in_proj_z.weight = assign(
                lat.in_proj_z.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_z.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_z.weight"),
            )
            lat.in_proj_b.weight = assign(
                lat.in_proj_b.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_b.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_b.weight"),
            )
            lat.in_proj_a.weight = assign(
                lat.in_proj_a.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_a.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_a.weight"),
            )

        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        block.norm1.weight = assign(
            block.norm1.weight,
            params[pkey(f"layers.{l}.input_layernorm.weight")],
            pkey(f"layers.{l}.input_layernorm.weight"),
        )

        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[pkey(f"layers.{l}.mlp.gate_proj.weight")],
            pkey(f"layers.{l}.mlp.gate_proj.weight"),
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[pkey(f"layers.{l}.mlp.up_proj.weight")],
            pkey(f"layers.{l}.mlp.up_proj.weight"),
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[pkey(f"layers.{l}.mlp.down_proj.weight")],
            pkey(f"layers.{l}.mlp.down_proj.weight"),
        )
        block.norm2.weight = assign(
            block.norm2.weight,
            params[pkey(f"layers.{l}.post_attention_layernorm.weight")],
            pkey(f"layers.{l}.post_attention_layernorm.weight"),
        )

    model.final_norm.weight = assign(
        model.final_norm.weight,
        params[pkey("norm.weight")],
        pkey("norm.weight"),
    )

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    elif pkey("lm_head.weight") in params:
        model.out_head.weight = assign(model.out_head.weight, params[pkey("lm_head.weight")], pkey("lm_head.weight"))
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")
# 中文注释:以下开始真正从 HuggingFace Hub 下载 Qwen3.5-0.8B 的
# safetensors 权重分片,并调用上面定义的 load_weights_into_qwen3_5
# 把权重灌入模型。
# 注意:这里 import 的 hf_hub_download 要到下一个代码单元(加载分词器)
# 才会被调用,两处共用同一次 import,属于 notebook 里"跨单元共享导入"
# 的常见写法,并非本单元遗漏的死代码。
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

repo_id = "Qwen/Qwen3.5-0.8B"
local_dir = Path(repo_id).parts[-1]

# 先下载/定位 safetensors 分片索引文件(model.safetensors.index.json),
# 它记录了每个参数 key 具体存放在哪个分片文件里。
repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

# 依次加载所有分片文件(可能有多个 .safetensors 文件),
# 合并成一个完整的 "参数 key -> 张量" 字典。
weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

# 用合并好的权重字典填充模型,搬到目标 device 上;
# 加载完成后释放 weights_dict 占用的内存/显存(权重已经拷贝进模型参数里,
# 不再需要这份中间副本)。
load_weights_into_qwen3_5(model, QWEN3_5_CONFIG, weights_dict)
model.to(device)
del weights_dict

4. Load tokenizer

In [ ]:
# 中文注释:Qwen3_5Tokenizer —— 基于 tokenizers 库(BPE)的分词器封装,
# 额外处理 Qwen 系列的特殊 token(如 <|im_start|>/<|im_end|>、
# <think>/</think> 等)以及 Qwen3.5 的对话模板(chat template)拼接。
import re
from tokenizers import Tokenizer


# 中文注释:_SPECIALS 列出所有需要被当作"整体 token"而不是走 BPE
# 切分的特殊标记;_SPLIT_RE 用于在编码文本前,先把这些特殊标记从
# 普通文本中正则切分出来,分别处理。
class Qwen3_5Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    # tokenizer_file_path: tokenizer.json 路径;repo_id 用于判断该模型是
    # 基座(Base)模型还是对话(chat)模型,从而决定默认 eos 用哪个 token;
    # apply_chat_template/add_generation_prompt/add_thinking
    # 控制 encode() 是否自动套用下面 _wrap_chat 里的对话模板
    # (以及是否留出 <think> 推理区间,对应 Qwen3.5 的思考模式)。
    def __init__(
        self,
        tokenizer_file_path="tokenizer.json",
        repo_id=None,
        apply_chat_template=True,
        add_generation_prompt=False,
        add_thinking=False,
    ):
        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    # encode:若整段文本本身就是一个特殊 token(如单独传入
    # "<|endoftext|>"),直接返回其 id;否则按需套用对话模板,
    # 再用 _SPLIT_RE 把特殊 token 和普通文本切开分别处理
    # (普通文本片段交给底层 BPE tokenizer 编码)。
    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    # _wrap_chat:按 Qwen3.5 的对话模板拼出 "<|im_start|>user\n...
    # <|im_end|>\n" 结构;若需要生成提示(add_generation_prompt),
    # 再追加 "<|im_start|>assistant\n";是否留出可写入推理过程的
    # "<think>\n"(open,留给模型自己续写)还是直接给出
    # 空的 "<think>\n\n</think>\n\n"(不思考,直接回答),
    # 由 add_thinking 控制。
    def _wrap_chat(self, user_msg):
        # Mirrors Qwen3.5 chat_template behavior:
        # add_generation_prompt + thinking => "<think>\n"
        # add_generation_prompt + no thinking => empty think scaffold
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant\n"
            if self.add_thinking:
                s += "<think>\n"
            else:
                s += "<think>\n\n</think>\n\n"
        return s
# 中文注释:下载 tokenizer.json 文件,并用上面定义的 Qwen3_5Tokenizer
# 加载出来;这里显式开启 apply_chat_template + add_generation_prompt +
# add_thinking,即后续所有 encode() 调用都会自动套上对话模板并留出
# <think> 思考区间。
tokenizer_file_path = "Qwen3.5-0.8B/tokenizer.json"

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

tokenizer = Qwen3_5Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
prompt = "Give me a short introduction to large language models."
# 简单验证一下:把 prompt 编码成 id 再解码回文本,
# 检查对话模板/特殊 token 是否按预期拼接。

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Generate text

In [ ]:
# 中文注释:基于 KV 缓存的流式自回归生成函数(贪心解码,greedy decoding,
# 每步直接取 argmax,不做采样)。
# 关键点:第一次调用 model(token_ids, cache=cache) 是"prefill"——
# 一次性把整段 prompt 喂进去,full_attention 层建立起完整的 (k, v)
# 缓存,linear_attention 层走分块并行路径并把最终状态写入 linear_cache;
# 之后每一步只把"上一步新生成的这 1 个 token"喂给模型
# (而不是把已生成的全部 token 重新喂一遍),full_attention 层依赖
# 缓存里的历史 key/value 做注意力,linear_attention 层则切换到
# 逐 token 递归(增量更新循环状态)的路径——这正是 KV 缓存能大幅加速
# 自回归生成的原因。
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()
    with torch.no_grad():
        # 为本次生成新建一份 KVCache(按层数初始化两类缓存容器),
        # 并重置模型内部的绝对位置计数器 current_pos,
        # 避免延续上一次生成遗留的位置状态。
        cache = KVCache(n_layers=model.cfg["n_layers"])
        model.reset_kv_cache()

        # Prime the cache with the initial context
        logits = model(token_ids, cache=cache)

        # 后续每一步:只对新生成的 1 个 token 做一次前向
        # (seq_len == 1),配合 cache 复用历史计算结果。
        for _ in range(max_new_tokens):
            next_token = torch.argmax(logits[:, -1], dim=-1, keepdim=True)

            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break

            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)
            # Feed only the new token to the model; cache handles history
            logits = model(next_token, cache=cache)
import time

prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

In [ ]:
# 中文注释:与上一个单元逻辑完全相同(同样调用带 KV 缓存的
# generate_text_basic_stream),只是换了一个更侧重推理/计算能力的
# 提示词(先打折、再加税,求最终价格与原价的差异),用于直观感受
# 模型的生成质量与(有 KV 缓存时的)生成速度。
import time

prompt = "A shop gives a 20% discount, then adds 10% tax. Is the final price higher or lower than the original? By how much?"

input_token_ids = tokenizer.encode(prompt)
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")